<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/basic_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/vishaljoshi24/dnd-concordia/main/examples/requirements.in' 'git+https://github.com/vishaljoshi24/dnd-concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
# !pip install gdm-concordia[ollama]

Dependencies

In [ ]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
API_TYPE = 'openai'
MODEL_NAME = 'gpt-5'
API_KEY = ''
DISABLE_LANGUAGE_MODEL = False

In [ ]:
model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
    api_key=API_KEY
)

In [ ]:
DISABLE_LANGUAGE_MODEL = False
if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [ ]:
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

D&D Agent Prefab

In [ ]:
import dataclasses
from concordia.typing import prefab as prefab_lib
from concordia.agents import entity_agent_with_logging
from concordia.components import agent as agent_components

@dataclasses.dataclass
class DnDAgent(prefab_lib.Prefab):
    description: str = "A custom agent that acts like a D&D player."

    def build(self, model, memory_bank):
        name = self.params.get("name", "Agent")
        goal = self.params.get("goal", "")
        dnd_class = self.params.get("class", "")
        dnd_race = self.params.get("race", "")
        hit_points = self.params.get("hit_points", 10)
        abilities = self.params.get("abilities", {})
        modifiers = self.params.get("modifiers", {})
        proficiency_bonus = self.params.get("proficiency_bonus", 2)
        skills = self.params.get("skills", {})
        equipment = self.params.get("equipment", [])
        backpack = self.params.get("inventory", [])
        background = self.params.get("background", "")
        personality_traits = self.params.get("personality", "")
        ideals = self.params.get("ideals", "")
        bonds = self.params.get("bonds", "")
        flaws = self.params.get("flaws", "")

        # Create components
        memory = agent_components.memory.AssociativeMemory(
            memory_bank=memory_bank)
        instructions = agent_components.instructions.Instructions(
            agent_name=name)
        observation = agent_components.observation.LastNObservations(
            history_length=50)

        components = {
            agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: memory,
            "Instructions": instructions,
            agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: observation,
        }

        act_component = agent_components.concat_act_component.ConcatActComponent(
            model=model,
            component_order=list(components.keys()),
        )

        return entity_agent_with_logging.EntityAgentWithLogging(
            agent_name=name,
            act_component=act_component,
            context_components=components,
        )

# Register custom prefab
prefabs["my_custom__Entity"] = DnDAgent()

In [ ]:
prefabs['dndagent__Entity'] = DnDAgent()

Context, Shared Memories, Player Agents and Game Masters

In [ ]:
PLACE = 'Wizards Tower Brewing Company, Forgotten Realms'
CONTEXT = ('This D&D short scenario specifically concerns a craft brewery.',
'It is set in the Wizards Tower Brewing Company and is in dire need of help.',
'A band of reliable, affordable adventurers are needed to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
'For the duration of the scenario, only Alice, Bob and the brewery owner are in the brewery',
'At the beginning of this adventure Alice and Bob',
'meet in the Wizard Tower Brewing Company. These two adventurers',
'DO NOT know each other AT FIRST and need to get to know each other.',
'The brewery owner hands out pints of Ale to Alice and Bob as they get to know each other.',
)

SHARED_MEMORIES = ('The brewery owner gives Alice and Bob a run-down of the task with some background:',
'- The business has been doing well and looks to expand its operations.',
'But first the beer cellar needed to be expanded.',
'- Workmen that he sent down into the cellar to expand it were attacked by',
'large black rats which came out of the wall they were digging out.',
'- The workmen escaped unharmed but the cellars are unusable which is bad for business.',
'- The rats may have something to do with the wizard\'s tower on the site which the brewery',
'takes its name from.',
'The brewery owner then lays out the terms of the task:',
'- The party must dispose of the rats.',
'- They must discover the origin of the rats and make sure they permanently',
'stop the infestation.',
'- They will each be rewarded 25 gold coins.',)

In [ ]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Alice',
            'goal': 'To collaborate with Bob in completing the task given by the brewery owner',
            'class': 'Warlock',
            'race': 'Mountain Dwarf',
            'hit_points': 11,
            'abilities': {
                'strength': 15,
                'dexterity': 12,
                'constitution':16,
                'intelligence': 8,
                'wisdom': 15,
                'charisma': 8,
            },
            'modifiers': {
                'strength': 2,
                'dexterity': 1,
                'constitution': 3,
                'intelligence': 0,
                'wisdom': 2,
                'charisma': -1,
            },
            'proficiency_bonus': 2,
            'skills': {
                'acrobatics': 1,
                'animal_handling': 2,
                'arcana': 2,
                'athletics': 2,
                'deception': -1,
                'history': 2,
                'insight': 4,
                'intimidation': -1,
                'investigation': 0,
                'medicine': 2,
                'nature': 2,
                'perception': 2,
                'performance': -1,
                'persuasion': -1,
                'religion': 0,
                'sleight_of_hand': 1,
                'stealth': 1,
                'survival': 2,
            },
            'equipment': [
                'common clothes',
                'crossbow bolts',
                'light crossbow',
                'dagger',
                'dagger',
                'ink',
                'leather armour',
                'sickle',
                'small knife',
                'staff',
                'battleaxe',
                'warhammer',
                'smith tools',
            ],
            'backpack': [
                'crowbar',
                'hammer',
                'piton',
                'rations',
                'hempen rope',
                'tinderbox',
                'torch',
                'waterskin',
                ],
            'background': 'sage',
            'personality_traits': 'There is nothing I like more than a good mystery. I am willing to listen to every side of an argument before I make my own judgement.',
            'ideals':'No limits. Nothing should fetter the infinite possibility inherent in all existence.',
            'bonds': 'I have been searching my whole life for the answer to a certain question.',
            'flaws': 'I am easily distracted by the promise of information.'
            },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Bob',
            'goal': 'To collaborate with Bob in completing the task given by the brewery owner',
            'class': 'Bard',
            'race': 'Human',
            'hit_points': 10,
            'abilities': {
                'strength': 10,
                'dexterity': 9,
                'constitution':15,
                'intelligence': 13,
                'wisdom': 13,
                'charisma': 15,
            },
            'modifiers': {
                'strength': 0,
                'dexterity': -1,
                'constitution': 2,
                'intelligence': 1,
                'wisdom': 1,
                'charisma': 2,
            },
            'proficiency_bonus': 2,
            'skills': {
                'acrobatics': -1,
                'animal_handling': 3,
                'arcana': 1,
                'athletics': 0,
                'deception': 2,
                'history': 1,
                'insight': 3,
                'intimidation': 2,
                'investigation': 1,
                'medicine': 1,
                'nature': 1,
                'perception': 3,
                'performance': 4,
                'persuasion': 2,
                'religion': 3,
                'sleight_of_hand': -1,
                'stealth': -1,
                'survival': 3,
            },
            'equipment': [
                'dagger',
                'dagger',
                'leather armour',
                'lute',
            ],
            'backpack': [
                'bedroll',
                'bell',
                'lantern',
                'costume',
                'mirror',
                'oil',
                'rations',
                'tinderbox',
                'waterskin',
                ],
            'background': 'folk hero',
            'personality_traits': 'If someone is in trouble, I am always ready to lend help. I judge people by their actions, not their words.',
            'ideals': 'Respect. People deserve to be treated with dignity and respect.',
            'bonds': 'I protect those who cannot protect themselves.',
            'flaws': 'I have a weakness for the vices of the city, especially hard drink.',
            },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'acting_order': 'game_master_choice',
            'shared_memories': SHARED_MEMORIES,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='situated_in_time_and_place__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            'start_time': 'Afternoon',
            'locations': PLACE,
        },
    )
]

In [ ]:
# from concordia.environment.engines import sequential

# engine = sequential.Sequential()

In [ ]:
config = prefab_lib.Config(
    default_premise=(
        CONTEXT
    ),
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

Instantiate and Run Simulation

In [ ]:
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
    # engine=engine,
)

In [ ]:
results_log = runnable_simulation.play(max_steps=3)

Logging

In [ ]:
from concordia.utils.structured_logging import AIAgentLogInterface

interface = AIAgentLogInterface(results_log)

In [ ]:
overview = interface.get_overview()

In [ ]:
print(overview)

In [ ]:
actions = interface.get_component_values()
for action in actions:
    print(f"Step {action['step']} [{action['entity_name']}]: {action['value']}")

# Filter by entity and step range
interface.get_component_values(
    entity_name='Alice', step_range=(1, 5),
)

In [ ]:
import json
from concordia.utils.structured_logging import SimulationLog

# Save
with open('/simulation_log.json', 'w') as f:
    f.write(results_log.to_json())

In [ ]:
log = SimulationLog.from_json(open('/simulation_log.json').read())
interface = AIAgentLogInterface(log)